# Retail Store Smart Inventory Management and Demand Forecasting
## Final Reproducible Google Colab Notebook

**Dataset:** FreshRetailNet-50K (Dingdong-Inc)  
**Final model:** Stock-aware XGBoost Regressor  
**Target:** Next-day product sales (`sale_amount`)

### Validated development result
- Naïve baseline MAE: **0.5746**
- Random Forest stock-aware MAE: **0.4396**
- XGBoost stock-aware MAE: **0.4383**
- XGBoost MAE improvement vs baseline: **23.72%**

> These metrics are from the validated 150-product development subset. They should not be presented as full-dataset performance.


## Workflow
1. Load the published FreshRetailNet-50K dataset.
2. Use a configurable product subset to control Colab RAM.
3. Build leakage-safe demand and historical-stockout features.
4. Split chronologically (80/20).
5. Compare a naïve baseline, Random Forest, and GPU XGBoost.
6. Inspect feature importance.
7. Generate forecast-driven inventory risk/replenishment priority.
8. Export the trained model and metadata.

**Inventory limitation:** the source data does not provide a complete physical inventory ledger, supplier lead time, ordering cost, or holding cost. The downstream inventory layer is therefore decision support, not an exact EOQ calculation.


In [ ]:
# 1. Install dependencies
!pip -q install -U datasets xgboost scikit-learn pandas numpy pyarrow

import gc
import json
import warnings
import numpy as np
import pandas as pd
import xgboost as xgb

from datasets import load_dataset
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
PRODUCT_LIMIT = 150

RF_ESTIMATORS = 150
XGB_ESTIMATORS = 250
XGB_MAX_DEPTH = 7
XGB_LEARNING_RATE = 0.05
XGB_SUBSAMPLE = 0.80
XGB_COLSAMPLE = 0.90

print("XGBoost:", xgb.__version__)


In [ ]:
# 2. Load FreshRetailNet-50K
dataset = load_dataset("Dingdong-Inc/FreshRetailNet-50K")
train = dataset["train"]
eval_df = dataset["eval"]

print("Train rows:", train.num_rows)
print("Eval rows :", eval_df.num_rows)
print("Columns:", train.column_names)


In [ ]:
# 3. Build the reproducible development subset
pilot = train.filter(lambda x: x["product_id"] < PRODUCT_LIMIT).to_pandas()

pilot["dt"] = pd.to_datetime(pilot["dt"])
pilot = pilot.sort_values(["store_id","product_id","dt"]).reset_index(drop=True)

print("Pilot rows:", len(pilot))
print("Stores:", pilot["store_id"].nunique())
print("Products:", pilot["product_id"].nunique())
print("Dates:", pilot["dt"].min(), "to", pilot["dt"].max())


## Feature engineering

All predictors use only information available before the prediction date.

- `lag_1`: previous-day sales
- `lag_7`: sales seven days earlier
- `rolling_7`: preceding 7-day mean sales
- `stockout_lag_1`: previous-day stockout hours
- `stockout_lag_7`: stockout hours seven days earlier
- `stockout_rolling_7`: preceding 7-day stockout hours
- `target_next_day`: next day's sales

The rolling features use `shift(1)` to prevent leakage.


In [ ]:
# 4. Create target and leakage-safe features
GROUPS = ["store_id","product_id"]
grp = pilot.groupby(GROUPS, sort=False)

pilot["lag_1"] = grp["sale_amount"].shift(1)
pilot["lag_7"] = grp["sale_amount"].shift(7)
pilot["rolling_7"] = grp["sale_amount"].transform(
    lambda x: x.shift(1).rolling(7).mean()
)

pilot["stockout_lag_1"] = grp["stock_hour6_22_cnt"].shift(1)
pilot["stockout_lag_7"] = grp["stock_hour6_22_cnt"].shift(7)
pilot["stockout_rolling_7"] = grp["stock_hour6_22_cnt"].transform(
    lambda x: x.shift(1).rolling(7).sum()
)

pilot["target_next_day"] = grp["sale_amount"].shift(-1)

FEATURES = [
    "lag_1","lag_7","rolling_7",
    "stockout_lag_1","stockout_lag_7","stockout_rolling_7"
]

pilot = pilot.dropna(subset=FEATURES + ["target_next_day"]).reset_index(drop=True)
print("Rows after feature engineering:", len(pilot))


In [ ]:
# 5. Chronological 80/20 split
cutoff = pilot["dt"].quantile(0.80)

train_model = pilot[pilot["dt"] <= cutoff].copy()
test_model = pilot[pilot["dt"] > cutoff].copy()

X_train = train_model[FEATURES]
y_train = train_model["target_next_day"]
X_test = test_model[FEATURES]
y_test = test_model["target_next_day"]

print("Training rows:", len(train_model))
print("Testing rows :", len(test_model))
print("Cutoff:", cutoff)


In [ ]:
# 6. Naïve previous-day baseline
baseline_pred = test_model["lag_1"]

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))

print(f"Baseline MAE : {baseline_mae:.4f}")
print(f"Baseline RMSE: {baseline_rmse:.4f}")


In [ ]:
# 7. Random Forest benchmark
rf = RandomForestRegressor(
    n_estimators=RF_ESTIMATORS,
    max_depth=10,
    min_samples_leaf=3,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))

print(f"Random Forest MAE : {rf_mae:.4f}")
print(f"Random Forest RMSE: {rf_rmse:.4f}")

del rf, rf_pred
gc.collect()


In [ ]:
# 8. GPU XGBoost — final candidate
xgb_model = xgb.XGBRegressor(
    n_estimators=XGB_ESTIMATORS,
    max_depth=XGB_MAX_DEPTH,
    learning_rate=XGB_LEARNING_RATE,
    subsample=XGB_SUBSAMPLE,
    colsample_bytree=XGB_COLSAMPLE,
    objective="reg:squarederror",
    eval_metric="mae",
    tree_method="hist",
    device="cuda",
    random_state=RANDOM_STATE
)

xgb_model.fit(X_train, y_train, verbose=False)
xgb_pred = xgb_model.predict(X_test)

xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))

print(f"XGBoost MAE : {xgb_mae:.4f}")
print(f"XGBoost RMSE: {xgb_rmse:.4f}")


In [ ]:
# 9. Model comparison
results = pd.DataFrame({
    "Model": ["Naive Baseline","Random Forest (Stock-Aware)","XGBoost (Stock-Aware)"],
    "MAE": [baseline_mae, rf_mae, xgb_mae],
    "RMSE": [baseline_rmse, rf_rmse, xgb_rmse]
}).round(4)

display(results)

print("XGBoost improvement vs baseline:",
      round((baseline_mae - xgb_mae) / baseline_mae * 100, 2), "%")
print("XGBoost improvement vs RF:",
      round((rf_mae - xgb_mae) / rf_mae * 100, 2), "%")


In [ ]:
# 10. Feature importance
importance = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": xgb_model.feature_importances_
}).sort_values("Importance", ascending=False)

importance["Importance (%)"] = (importance["Importance"] * 100).round(2)
display(importance[["Feature","Importance (%)"]].reset_index(drop=True))


## Forecast-driven inventory decision support

The following is deliberately conservative. It uses the forecast plus recent stockout pressure to prioritize replenishment actions.

The safety factor and risk thresholds are **project assumptions**, not learned inventory-economic parameters.


In [ ]:
# 11. Inventory decision support
decision_df = test_model[
    ["dt","store_id","product_id","rolling_7","stockout_rolling_7"]
].copy()

decision_df["predicted_next_day_demand"] = np.clip(xgb_pred, 0, None)
decision_df["predicted_7day_demand"] = decision_df["predicted_next_day_demand"] * 7

SAFETY_FACTOR = 0.50

decision_df["demand_variability"] = decision_df["rolling_7"].fillna(0)
decision_df["safety_buffer"] = decision_df["demand_variability"] * SAFETY_FACTOR
decision_df["recommended_stock_coverage"] = (
    decision_df["predicted_7day_demand"] + decision_df["safety_buffer"]
)

decision_df["recent_stockout_hours"] = decision_df["stockout_rolling_7"].fillna(0)

def classify_risk(hours):
    if hours >= 24:
        return "HIGH"
    if hours >= 8:
        return "MEDIUM"
    return "LOW"

def recommendation(risk):
    if risk == "HIGH":
        return "REPLENISH NOW"
    if risk == "MEDIUM":
        return "MONITOR / REPLENISH"
    return "NO IMMEDIATE ACTION"

decision_df["inventory_risk"] = decision_df["recent_stockout_hours"].apply(classify_risk)
decision_df["recommended_action"] = decision_df["inventory_risk"].apply(recommendation)

numeric_cols = [
    "predicted_next_day_demand","predicted_7day_demand",
    "demand_variability","safety_buffer",
    "recommended_stock_coverage","recent_stockout_hours"
]
decision_df[numeric_cols] = decision_df[numeric_cols].round(2)

print("Risk distribution:")
display(decision_df["inventory_risk"].value_counts())

print("Recommendation distribution:")
display(decision_df["recommended_action"].value_counts())

display(decision_df.head(20))


In [ ]:
# 12. Validate whether predicted risk aligns with actual stockout pressure
risk_validation = decision_df[
    ["dt","store_id","product_id","inventory_risk"]
].copy()

actual = test_model[
    ["dt","store_id","product_id"]
].copy()

# Pull the actual current-day stockout hours from the original pilot.
# This is only used for evaluation, never as an input to the forecast.
actual["actual_stockout_hours"] = pilot.loc[
    test_model.index, "stock_hour6_22_cnt"
].to_numpy()

risk_validation = risk_validation.merge(
    actual,
    on=["dt","store_id","product_id"],
    how="left"
)

display(
    risk_validation.groupby("inventory_risk")["actual_stockout_hours"]
    .agg(rows="count", mean="mean", median="median", max="max")
    .round(2)
)

display(
    risk_validation.groupby("inventory_risk")["actual_stockout_hours"]
    .apply(lambda x: (x > 0).mean() * 100)
    .round(2)
    .rename("actual_stockout_rate_percent")
)


## 13. Export model and metadata

The model is saved in XGBoost's native JSON format. The metadata file records the feature order, target, prediction horizon, validation metrics, and project assumptions.


In [ ]:
# 14. Export artifacts
MODEL_PATH = "/content/xgboost_stock_aware.json"
FEATURE_PATH = "/content/model_features.json"

xgb_model.save_model(MODEL_PATH)

metadata = {
    "model_type": "XGBoost Regressor",
    "target": "target_next_day",
    "features": FEATURES,
    "prediction_horizon": "next_day",
    "product_limit_used_for_validation": PRODUCT_LIMIT,
    "validation_cutoff": str(cutoff.date()),
    "validation_mae": round(float(xgb_mae), 4),
    "validation_rmse": round(float(xgb_rmse), 4),
    "baseline_mae": round(float(baseline_mae), 4),
    "baseline_rmse": round(float(baseline_rmse), 4),
    "improvement_vs_baseline_percent": round(
        float((baseline_mae - xgb_mae) / baseline_mae * 100), 2
    ),
    "inventory_safety_factor": SAFETY_FACTOR,
    "risk_thresholds_stockout_hours": {
        "LOW": "< 8",
        "MEDIUM": "8-23",
        "HIGH": ">= 24"
    }
}

with open(FEATURE_PATH, "w") as f:
    json.dump(metadata, f, indent=4)

print("Model:", MODEL_PATH)
print("Metadata:", FEATURE_PATH)


In [ ]:
# 15. Download the artifacts
from google.colab import files
files.download(MODEL_PATH)
files.download(FEATURE_PATH)


## Final reporting notes

- Report **MAE/RMSE**, not generic classification accuracy.
- State that the reported 23.72% improvement is from the validated 150-product development subset.
- Explain that recent demand history is the dominant predictor.
- Explain that lagged stockout history provides an additional inventory-related signal.
- Present replenishment as **forecast-driven decision support**, not exact EOQ/reorder quantity.
- Keep the chronological split; do not replace it with a random split.
